In [1]:
library(auk)
library(tidyverse)

library(magrittr)
library(httr)
library(data.table)
library(plotly)
# library(elevatr)
library(readr)
library(raster)
library(purrr)
library(dplyr)

library(sf)
library(terra)
# library(prism)
# library(rnaturalearth)
library(dggridR)
library(lubridate)
library(hms)

library(gridExtra)

select <- dplyr::select
projection <- raster::projection

readRenviron("~/.Renviron")
R.home()

auk 0.9.0 is designed for EBD files downloaded after 2025-10-28. 
No EBD data directory set, see ?auk_set_ebd_path to set EBD_PATH 
eBird taxonomy version:  2025

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘magrittr’


The following object is masked from ‘package:purrr’:

    set_names


The following object is masked from ‘package:tidyr’:

    extract



Attaching package: ‘data.table’


The following objects are masked from ‘package:lubridate’:

    hour, isoweek, isoyear, mday, minute, month

[1] "/usr/lib/R"

In [2]:
PROJECT_DIR = getwd()

In [ ]:
time_to_decimal <- function(x) {
    x <- as_hms(x)
    hour(x) + minute(x) / 60 + second(x) / 3600
}

combine_chunk_aggs <- function(files) {
  bind_rows(map(files, read_csv, show_col_types = FALSE)) %>%
    group_by(cell, week) %>%
    summarise(
      n_checklists = sum(n_checklists, na.rm = TRUE),
      n_detected   = sum(n_detected, na.rm = TRUE),
      tot_observed = sum(tot_observed, na.rm = TRUE),
      cell_ctr_lat = first(na.omit(cell_ctr_lat)),
      cell_ctr_lon = first(na.omit(cell_ctr_lon)),
      country_code = names(sort(table(country_code), decreasing = TRUE))[1],
      .groups = "drop"
    ) %>%
    mutate(
      det_freq = if_else(n_checklists > 0, n_detected / n_checklists, NA_real_)
    )
}


grid_res_km <- 50
dggs <- dgconstruct(spacing = grid_res_km, topology = "HEXAGON")


In [ ]:
prefix <- "all_raptors_20260821"
# prefix <- "ar_20260821"

split_count = 3

species.names <- c(
 "Accipiter_striatus",        # Sharp-shinned Hawk
 "Aquila_chrysaetos",         # Golden Eagle
 "Asio_flammeus",             # Short-eared Owl
 "Asio_otus",                 # Long-eared Owl
 "Astur_atricapillus",        # Northern Goshawk
 "Astur_cooperii",            # Cooper's Hawk
 "Athene_cunicularia",        # Burrowing Owl
 "Bubo_virginianus",          # Great Horned Owl
 "Buteo_jamaicensis",         # Red-tailed Hawk
 "Buteo_lineatus",            # Red-shouldered Hawk
 "Buteo_platypterus",         # Broad-winged Hawk
 "Buteo_regalis",             # Ferruginous Hawk
 "Buteo_swainsoni",           # Swainson's Hawk
 "Caracara_plancus",          # Crested Caracara
 "Circus_hudsonius",          # Northern Harrier
 "Elanoides_forficatus",      # STK
 "Elanus_leucurus",           # White-tailed Kite
 "Falco_columbarius",         # Merlin
 "Falco_peregrinus",          # Peregrine Falcon
 "Falco_sparverius",          # American Kestrel
 "Haliaeetus_leucocephalus",  # Bald Eagle
 "Ictinia_mississippiensis",  # MSK
 "Pandion_haliaetus",         # Osprey
 "Parabuteo_unicinctus",      # Harris's Hawk
 "Psiloscops_flammeolus",     # Flammulated Owl
 "Strix_varia"                # Barred Owl
)

for (species.name in species.names) {
    for (split_num in 1:split_count) {

        file.desc <- paste0(prefix, "_", species.name, "_", split_num)
        f_out_ebd_only <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, ".txt"))

        file.desc.sed <- paste0(prefix, "_", split_num)
        f_out_sed_only <- file.path(PROJECT_DIR, "output", "auk", paste0("sed_", file.desc.sed, ".txt"))

        print(paste0("Reading in data for species: ", species.name, ", split: ", split_num))
        ebd_only_df <- read_ebd(f_out_ebd_only, unique=TRUE, rollup=TRUE) # do not need to use auk_unique() when unique=TRUE passed here. Same for auk_rollup()
        sed_only_df <- read_sampling(f_out_sed_only, unique=TRUE)

        print(paste0("Zero-filling and cleaning, species: ", species.name, ", split: ", split_num))
        ebd_zf_df <- auk_zerofill(ebd_only_df, sampling_events = sed_only_df,
                                  collapse = TRUE)

        ebd_df <- ebd_zf_df |> 
        mutate(
            observation_count = if_else(observation_count == "X", 
                                        NA_character_, observation_count),
            observation_count = as.integer(observation_count),
            effort_distance_km = if_else(protocol_name == "Stationary", 
                                         0, effort_distance_km),
            hours_of_day = time_to_decimal(time_observations_started),
            year = year(observation_date),
            day_of_year = yday(observation_date)
        )

        print(paste0("Assigning hex grid cells for species: ", species.name, ", split: ", split_num))
        ebd_with_cells <- ebd_df |>
        mutate(cell = dgGEO_to_SEQNUM(dggs, longitude, latitude)$seqnum,
               week = isoweek(observation_date))

        # hexagons <- dgcellstogrid(dggs, unique(ebd_with_cells$cell)) %>% 
        #   st_as_sf()

        # hexagons$cell <- as.integer(hexagons$seqnum)
        # hexagons <- hexagons %>% select(cell, geometry)
        # hex_spatvector <- terra::vect(hexagons)

        # ebd_agg <- ebd_with_cells %>% 
        #   group_by(cell) %>% 
        #   summarize(n_checklists = n(),
        #             n_detected = sum(species_observed),
        #             det_freq = mean(species_observed),
        #             tot_observed = sum(observation_count),
        #             first_date = min(observation_date),
        #             last_date = max(observation_date),
        #             country_code = names(sort(table(country_code), decreasing = TRUE))[1]) %>% 
        #   ungroup() %>%
        #   mutate(cell_ctr_lat = dgSEQNUM_to_GEO(dggs, cell)$lat_deg,
        #          cell_ctr_lon = dgSEQNUM_to_GEO(dggs, cell)$lon_deg)

        print(paste0("Aggregating weekly data for species: ", species.name, ", split: ", split_num))
        ebd_agg_weekly <- ebd_with_cells %>% 
        group_by(cell, week) %>% 
        summarize(n_checklists = n(),
                  n_detected = sum(species_observed),
                  det_freq = mean(species_observed),
                  tot_observed = sum(observation_count),
                  country_code = names(sort(table(country_code), decreasing = TRUE))[1]) %>% 
        ungroup() %>%
        mutate(cell_ctr_lat = dgSEQNUM_to_GEO(dggs, cell)$lat_deg,
               cell_ctr_lon = dgSEQNUM_to_GEO(dggs, cell)$lon_deg)

        print(paste0("Writing aggregated weekly data for species: ", species.name, ", split: ", split_num))
        write_csv(ebd_agg_weekly, file.path(PROJECT_DIR, "output", paste0(file.desc, "_zf_clean_agg_weekly.csv")))

        # Release memory
        rm(list = c("ebd_only_df",
                    "sed_only_df",
                    "ebd_zf_df",
                    "ebd_df",
                    "ebd_with_cells",
                    "ebd_agg_weekly"))
    }

    print(paste0("Combining aggregated weekly data for species: ", species.name))
    file.descs <- paste0(prefix, "_", species.name, "_", seq_len(split_count))
    
    ebd_agg_weekly_comb <- combine_chunk_aggs(file.path(PROJECT_DIR, "output", paste0(file.descs, "_zf_clean_agg_weekly.csv")))
    write_csv(ebd_agg_weekly_comb, file.path(PROJECT_DIR, "output", paste0(prefix, "_", species.name, "_zf_clean_agg_weekly.csv")))
    ebd_agg_weekly_comb <- c() # clear memory

}

[1] "Combining aggregated weekly data for species: Asio_flammeus"
